# B1 — RDF Generation

Generates RDF from the CSVs produced by B0 using YARRRML/RML mapping.
Creates Fuseki dataset `nancy-museum`, uploads ontologies and generated triples.
Inserts artwork category types via SPARQL UPDATE (no GPT classification needed).

In [ ]:
import os
import sys
import subprocess
import pandas as pd

sys.path.insert(0, os.path.abspath('helpers'))
from fuseki_config import (
    graph_name, query_endpoint, update_endpoint,
    BIRD_ROOT, PROJECT_ROOT, setup_new_dataset, upload_file
)
from fuseki_queries import execute_sparql_query, execute_sparql_update, format_results

## 1. Run YARRRML Parser

In [ ]:
mappings_dir = PROJECT_ROOT / 'mappings'
yml_file = mappings_dir / 'rules_bird.yml'
rml_file = mappings_dir / 'rules_bird.rml.ttl'

print(f'YARRRML input:  {yml_file}')
print(f'RML output:     {rml_file}')

result = subprocess.run(
    ['yarrrml-parser', '-i', str(yml_file), '-o', str(rml_file)],
    capture_output=True, text=True, cwd=str(mappings_dir)
)
print(f'Return code: {result.returncode}')
if result.stdout:
    print(result.stdout[:500])
if result.returncode != 0 and result.stderr:
    print(f'STDERR: {result.stderr[:500]}')

## 2. Run RML Mapper

In [ ]:
rml_mapper = mappings_dir / 'rmlmapper-7.3.3-r374-all.jar'
output_ttl = BIRD_ROOT / 'files' / 'bird_data.ttl'
output_ttl.parent.mkdir(parents=True, exist_ok=True)

print(f'RML Mapper:  {rml_mapper}')
print(f'Output TTL:  {output_ttl}')

result = subprocess.run(
    ['java', '-jar', str(rml_mapper),
     '-m', str(rml_file),
     '-o', str(output_ttl),
     '-s', 'turtle'],
    capture_output=True, text=True, cwd=str(mappings_dir)
)
print(f'Return code: {result.returncode}')
if result.stdout:
    print(result.stdout[:500])
if result.returncode != 0 and result.stderr:
    print(f'STDERR: {result.stderr[:1000]}')

if output_ttl.exists():
    size_mb = output_ttl.stat().st_size / (1024*1024)
    print(f'\nGenerated {output_ttl.name}: {size_mb:.1f} MB')

## 3. Create Fuseki Dataset & Upload Ontologies

In [ ]:
setup_new_dataset()

## 4. Upload Generated RDF Data

In [ ]:
print(f'Uploading {output_ttl.name} to {graph_name}...')
success = upload_file(graph_name, output_ttl, content_type='text/turtle')
print(f'Upload {"succeeded" if success else "FAILED"}')

## 5. Insert Artwork Category Types via SPARQL

Categories are pre-computed in B0 via `artwork_categories.py`. We insert them as `poi:hasType` values.

In [ ]:
# Load artworks CSV to get category mapping
artworks = pd.read_csv(BIRD_ROOT / 'files' / 'csv' / 'artworks.csv')

# Build SPARQL INSERT DATA for category types
# The raw type is already stored via poi:hasType from RML mapping
# Now add the resolved category as an rdf:type
insert_lines = []
for _, row in artworks.iterrows():
    artwork_uri = f'poi:artwork_{row["image_id"]}'
    category = row['category']
    if category and category != 'Unknown':
        insert_lines.append(f'  {artwork_uri} a poi:{category} .')

if insert_lines:
    # Batch in groups of 500
    batch_size = 500
    total_inserted = 0
    for i in range(0, len(insert_lines), batch_size):
        batch = insert_lines[i:i+batch_size]
        triples = '\n'.join(batch)
        query = f"""
PREFIX poi: <http://w3id.org/trajecto/poi#>
INSERT DATA {{
{triples}
}}
"""
        success = execute_sparql_update(update_endpoint, query)
        if success:
            total_inserted += len(batch)
        else:
            print(f'  Batch {i//batch_size + 1} failed')

    print(f'Inserted {total_inserted} artwork category types')
else:
    print('No category types to insert')

## 5b. Insert Room and Era Metadata

Add `poi:hasRoom` and `poi:hasEra` triples for each artwork from `artworks.csv`.
Rooms (37) provide curatorial grouping. Eras (6 buckets) parsed from `poi:creationPeriod`.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('helpers'))
from artwork_categories import resolve_era, resolve_room

# Build room + era triples from artworks CSV
room_lines = []
era_lines = []
for _, row in artworks.iterrows():
    artwork_uri = f'poi:artwork_{row["image_id"]}'
    room = resolve_room(str(row.get('room_id', '')))
    if room:
        room_uri = f'poi:{room}'
        room_lines.append(f'  {artwork_uri} poi:hasRoom {room_uri} .')
    era = resolve_era(str(row.get('date', '')))
    if era and era != 'Unknown':
        era_lines.append(f'  {artwork_uri} poi:hasEra "{era}" .')

# Insert rooms
if room_lines:
    batch_size = 500
    total = 0
    for i in range(0, len(room_lines), batch_size):
        batch = room_lines[i:i+batch_size]
        triples = '\n'.join(batch)
        query = f"""
PREFIX poi: <http://w3id.org/trajecto/poi#>
INSERT DATA {{
{triples}
}}
"""
        if execute_sparql_update(update_endpoint, query):
            total += len(batch)
    print(f'Inserted {total} poi:hasRoom triples')

# Insert eras
if era_lines:
    total = 0
    for i in range(0, len(era_lines), batch_size):
        batch = era_lines[i:i+batch_size]
        triples = '\n'.join(batch)
        query = f"""
PREFIX poi: <http://w3id.org/trajecto/poi#>
INSERT DATA {{
{triples}
}}
"""
        if execute_sparql_update(update_endpoint, query):
            total += len(batch)
    print(f'Inserted {total} poi:hasEra triples')

## 6. Verify Triple Counts

In [ ]:
verification_queries = [
    ('Total triples', 'SELECT (COUNT(*) AS ?c) WHERE { ?s ?p ?o }'),
    ('Trajectories', 'SELECT (COUNT(DISTINCT ?t) AS ?c) WHERE { ?t a <http://www.datacron-project.eu/datAcron#Trajectory> }'),
    ('Positions', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a <http://www.datacron-project.eu/datAcron#RawPosition> }'),
    ('Persons', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a <http://w3id.org/trajecto#Person> }'),
    ('Artworks (POIs)', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a <http://w3id.org/trajecto#PointOfInterest> }'),
    ('Interactions', 'SELECT (COUNT(DISTINCT ?i) AS ?c) WHERE { ?i a <http://w3id.org/trajecto/poi#POIInteraction> }'),
    ('Stop positions', 'SELECT (COUNT(DISTINCT ?p) AS ?c) WHERE { ?p a <http://w3id.org/trajecto#StopPosition> }'),
]

print('=== Knowledge Graph Verification ===')
for label, query in verification_queries:
    result = execute_sparql_query(query_endpoint, query)
    df = format_results(result)
    count = df.iloc[0]['c'] if not df.empty else '?'
    print(f'  {label}: {count}')